<a href="https://colab.research.google.com/github/blankperson-cyber/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ML Task Framing: Predictive QoE & Perceptual Visibility Modeling for Immersive Media**


##Lane Overview

 **Lane:** Freestyle - Predictive Quality of Experience (QoE) and Perceptual Visibility Modeling

**Context:** As web ecosystems transition from flat 2D images to multi-dimensional spatial data (3D Gaussian Splatting, WebGL stereoscopic layers, volumetric video), the computational and network payload increases dramatically. This lane focuses on modeling the exact tipping point where heavy asset deployment compromises system performance and drops user retention, enabling adaptive immersive media streaming.


 ## 1. Task Type

 **Task Type:** **Regression + Binary Classification** (Hybrid)

 **Primary Task:** **Scoring** - Predict a continuous QoE score (0-1) representing perceived quality of experience for each immersive asset delivery

 **Secondary Task:** **Binary Classification** - Classify whether a given asset configuration will trigger a "retention drop event" (user abandonment) based on performance thresholds

 **Why this hybrid approach:**
 - **Scoring** allows fine-grained optimization: we need to know *how much* QoE degrades as asset complexity increases, not just whether it degrades
 - **Classification** creates actionable thresholds: production systems need binary decisions (serve vs. degrade) at runtime
 - The regression output feeds into the classification boundary, creating a continuous risk surface

 **ML Loop Position:**
 ```
 Asset Characteristics → Performance Signals → QoE Prediction → Adaptive Delivery Decision → Feedback Collection
 ```

## 2. Target or Proxy

**Direct Target:** Perceptual Quality Score (PQS) - 0 to 1, where 1 = visually lossless experience
**Proxy Variables (what we'll actually predict):**



---
**Frame Drop Rate** | % of frames dropped during playback | Directly impacts smoothness, a core QoE dimension

---

 **Time-to-Interactive (TTI)** | Time until asset is fully interactive | Users perceive "slowness" as poor quality

 ---

**Pixel Error Rate** | Difference between rendered and expected visual quality | Correlates with perceptual distortion

 ---

**User Engagement Decay** | Drop in interaction rate vs. baseline | Behavioral signal of frustration

 ---

**Bounce Risk Score** | Probability user leaves before full asset loads | Retention proxy

 ---


**Composite Target Calculation:**
```python
QoE_Score = w1 * (1 - normalized_frame_drop) +
  w2 * (1 - normalized_TTI) +
  w3 * (1 - normalized_pixel_error) +
  w4 * (1 - normalized_engagement_decay)
  
  Where: w1 = 0.25, w2 = 0.25, w3 = 0.25, w4 = 0.25 (equal weighting initially)

   ```
**Validation:** Compare against user-reported satisfaction surveys (NPS) and controlled user studies with varying asset complexities.

## 3. Success Metric

**Primary Metric: Root Mean Squared Error (RMSE)**

* Target: RMSE < 0.15 on a 0-1 scale
* Why: Penalizes large errors heavily, critical for user experience

**Secondary Metrics:**

1. Mean Absolute Error (MAE) - Interpretable error magnitude
2. R² Score - Variance explained in QoE
3. Classification F1-Score - For retention drop prediction
4. Precision@K - For top-k risky assets identification

**Business Metrics:**
* User retention improvement: +5% for high-risk assets
* Engagement increase: +10% for adaptive delivery
* Bandwidth savings: 20% without quality degradation

In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists("/content/flyrank-ml-internship-starter"):
    %cd /content/flyrank-ml-internship-starter
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

print("Loading FlyRank starter data...")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"✅ Data loaded successfully!")
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
df.head()


Loading FlyRank starter data...


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 5. Why ML Beats a Fixed Rule Here
**Complexity and Non-linearity:**
* QoE is a complex, non-linear function of multiple interacting factors
* Fixed rules can't capture interaction effects (e.g., high bandwidth + high latency ≠ good QoE)
* The relationship between technical metrics and perceived quality is highly context-dependent

**User Behavior Variance:**

* Different users have different tolerance thresholds for degradation
* Fixed rules assume uniform user preferences
* ML can learn user-specific patterns and preferences

**Dynamic Environment:**

* Network conditions, device capabilities, and asset types are constantly evolving
* Fixed rules become outdated quickly
* ML models can continuously adapt to new patterns

**Hidden Correlations:**

* Complex patterns in the data are not obvious to human developers
* Example: Certain combinations of metrics that don't individually look problematic might indicate poor QoE
* ML can discover these hidden relationships automatically

**Continuous Improvement:**

* Fixed rules are static and require manual updates
* ML models can learn from new data and improve over time
* The system can automatically adapt to changing conditions

Example: If fixed rule →
```
If frame_drop_rate > 0.1 OR latency > 100ms:
    degrade_quality()
Else:
    serve_full()

```

**Issues with fixed rule:**

* Misses complex interactions (e.g., high latency + low frame drops might still be bad)
* Cannot adapt to different asset types (3D Gaussian Splatting vs. WebGL)
* No feedback loop to learn from user behavior
* Binary decision (full vs. degrade) with no middle ground

**With ML:**

* Continuous QoE score for fine-grained decisions
* Learns optimal tradeoffs between all factors
* Adapts to different asset types, networks, and users
* Can provide probabilistic risk assessments
